# Nimbus Silver to Gold
Attach `nimbus_iq_gold` as default. Run Bronze to Silver first; the shared `medallion.py` module must remain in Bronze Files/src/data_generator.
Preserves semantic-model facts, dimensions, and causal bridges, and adds a calendar, hourly telemetry, daily health, incident impact, and dispatch summaries. All times use UTC.
Impact segments come from owning accounts (Consumer becomes Residential). Affected customers are CRM customers on impacted accounts, not an invented customer-subscription relationship. Cases and exposure are aggregated separately to avoid fanout.
Dispatch demo weights: severity High/Medium/other = 40/20/10; impacted services = min(25, count/20); incident exposure = min(25, CAD/1000); highest impacted account tier = Enterprise 10, Premium 5, other 0. Incident exposure repeats on each work order and must not be summed across work orders.
Existing outputs are protected unless `allow_replace = True` explicitly authorizes a full refresh. Writes are per-table, not a cross-table transaction. No local Python setup is required.


In [ ]:
bronze_lakehouse_name = 'nimbus_iq_bronze'
silver_lakehouse_name = 'nimbus_iq_silver'
target_lakehouse_name = 'nimbus_iq_gold'
source_workspace_id = ''
source_schema = 'auto'
allow_replace = False


In [ ]:
import importlib.util
import json
import sys
from pathlib import Path
from tempfile import TemporaryDirectory
from pyspark.sql import functions as fn

context = notebookutils.runtime.context
workspace_id = context['currentWorkspaceId']
source_workspace_id = source_workspace_id or workspace_id
target = notebookutils.lakehouse.get(target_lakehouse_name, workspace_id)
if str(context.get('defaultLakehouseId')) != str(target.id) or str(context.get('defaultLakehouseWorkspaceId')) != str(workspace_id):
    raise ValueError(f'Attach {target_lakehouse_name} as default and restart the Spark session.')
bronze = notebookutils.lakehouse.get(bronze_lakehouse_name, source_workspace_id)
silver = notebookutils.lakehouse.get(silver_lakehouse_name, source_workspace_id)
endpoint = notebookutils.conf.get('trident.onelake.endpoint').replace('https://', '').rstrip('/')
bronze_root = f'abfss://{source_workspace_id}@{endpoint}/{bronze.id}'
silver_root = f'abfss://{source_workspace_id}@{endpoint}/{silver.id}'
with TemporaryDirectory(prefix='nimbus_medallion_') as directory:
    module_path = Path(directory) / 'medallion.py'
    copied = notebookutils.fs.cp(f'{bronze_root}/Files/src/data_generator/medallion.py', module_path.as_uri())
    if copied is False or not module_path.exists():
        raise FileNotFoundError('Upload medallion.py to Bronze Files/src/data_generator first.')
    spec = importlib.util.spec_from_file_location('nimbus_medallion', module_path)
    medallion = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = medallion
    spec.loader.exec_module(medallion)
spark.conf.set('spark.sql.session.timeZone', 'UTC')
print(f'Source: {silver_lakehouse_name}; destination: {target_lakehouse_name}')


In [ ]:
silver_tables = medallion.read_layer(spark, notebookutils, silver_root, 'silver', source_schema)
gold_tables = medallion.build_gold(silver_tables)
primary_incident_id = 'INC-2026-ONT-001'
if silver_tables['incident'].where(fn.col('incident_id') == primary_incident_id).count() != 1:
    raise ValueError('The golden incident is missing from Silver.')
print('PASS: Gold impact, support-case, SLA, and investigation totals reconcile to Silver')


In [ ]:
row_counts = medallion.write_layer(spark, gold_tables, allow_replace)
stored_sources = {source: spark.table(target) for target, source in medallion.GOLD_SOURCES.items()}
medallion.validate_silver(stored_sources)
print(json.dumps({'layer': 'gold', 'status': 'PASS', 'tables': row_counts}, sort_keys=True))
display(spark.table('agg_customer_impact_by_incident').where(fn.col('incident_id') == primary_incident_id))
display(spark.table('agg_dispatch_priority').where(fn.col('incident_id') == primary_incident_id).orderBy(fn.col('dispatch_priority_score').desc()))
